# Example: Gradient-based optimisation of a cross-resonance gate between two transmons

In [ ]:
import itertools

import matplotlib.pyplot as plt

import numpy as np

from cthree.model.GeneratorDrive import GeneratorDrive
from cthree.propagation.Propagation import Propagation
from cthree.signal.Device import CosToneErf
from cthree.signal.SimpleGenerator import CosGenerator

np.set_printoptions(linewidth=400)

from cthree.OptimisationMap import OptimisationMap
from cthree.Quantity import Quantity
from cthree.measurement.UnitaryFidelity import UnitaryFidelity
from cthree.model.Coupling import Coupling
from cthree.optimisers.ScipyOptimiserGradient import ScipyOptimiserGradient
from cthree.propagation.ScipyExpmGOAT import ScipyExpmGOAT

from cthree.model.ClosedModel import ClosedModel
from cthree.model.CompositeHamiltonian import CompositeHamiltonian
from cthree.model.Transmon import Transmon

The sytem consists of two coupled transmons with three levels each. We fix the transmon frequency and anharmonicity to values that don't have any unwanted frequency collisions. The coupling strength is fixed as well. These parameters have to be specified as Quantites with a range, but we will not pass them to the optimised in order to keep them fixed. Additionally, the first transmon is driven at the frequency of the second one to apply a cross-resonance (CR) gate. The second transmon is driven to fix the phases of the gate. 

Here the tone values are set such that the optimisation process is fast. Generally with a lot of parameters `ScipyExpmGOAT` (in its current form), can take considerably long time. 

In [ ]:
tone1 = CosToneErf()
tone1.amplitude = Quantity(
    190e6 * 2*np.pi, 
    min_value=1e5 * 2*np.pi, 
    max_value=250e6 * 2*np.pi, 
    name="Amp", 
    unit="Hz", 
    twoPi=True)

tone1.frequency = Quantity(
    6.0e9 * 2*np.pi, 
    min_value=5.5e9 * 2*np.pi, 
    max_value=6.20e9 * 2*np.pi, 
    name="Freq", 
    unit="Hz", 
    twoPi=True)

t_final = 150e-9
tone1.t_final = Quantity(
    t_final,
    min_value=10e-9,
    max_value=200e-9,
    unit="s",
    name="Gate time",
)

tone2 = CosToneErf()
tone2.amplitude = Quantity(
    9.18e6 * 2*np.pi, 
    min_value=1e5 * 2*np.pi, 
    max_value=250e6 * 2*np.pi, 
    name="Amp", 
    unit="Hz", 
    twoPi=True)

tone2.frequency = Quantity(
    6.0002e9 * 2*np.pi, 
    min_value=5.5e9 * 2*np.pi, 
    max_value=6.20e9 * 2*np.pi, 
    name="Freq", 
    unit="Hz", 
    twoPi=True)

tone2.phase = Quantity(
    value=-0.39720756,
    min_value=-np.pi,
    max_value=np.pi,
    unit="rad",
    name="Phase",
)

tone2.t_final = Quantity(
    t_final,
    min_value=10e-9,
    max_value=200e-9,
    unit="s",
    name="Gate time",
)

print(tone1.getParameters(), tone2.getParameters())
generator1 = CosGenerator(devices=[tone1])
drive1 = GeneratorDrive(generator1, isLongitudinal=False)

generator2 = CosGenerator(devices=[tone2])
drive2 = GeneratorDrive(generator2, isLongitudinal=False)

transmon1 = Transmon(
    dimension=3,
    frequency=Quantity(5.5e9* 2*np.pi, 5.2e9* 2*np.pi, 5.9e9* 2*np.pi, "Hz", "Transmon 1 frequency", twoPi=True),
    anharmonicity=Quantity(-240e6* 2*np.pi, -250e6* 2*np.pi, -190e6* 2*np.pi, "Hz", "Transmon 1 anharmonicity", twoPi=True),
    drives=[drive1]
)
transmon2 = Transmon(
    dimension=3,
    frequency=Quantity(6.0e9* 2*np.pi, 5.9e9* 2*np.pi, 6.1e9* 2*np.pi, "Hz", "Transmon 2 frequency", twoPi=True),
    anharmonicity=Quantity(-200e6, -210e6, -190e6, "Hz", "Transmon 2 anharmonicity", twoPi=True),
    drives=[drive2]
)
coupling = Coupling(
    [transmon1, transmon2],
    isLongitudinal=False,
    coefficient=Quantity(25e6* 2*np.pi, 10e6* 2*np.pi, 60e6* 2*np.pi, "Hz", "Coupling strength", twoPi=True)
)
hamiltonian = CompositeHamiltonian([transmon1, transmon2], [coupling])

In [ ]:
(transmon2.getFrequency().getValue() - transmon1.getFrequency().getValue())/(2 * np.pi)

In [ ]:
times = np.linspace(0, t_final, 201)
signal1 = generator1.generateSignal(times)
signal2 = generator2.generateSignal(times)

plt.figure()
plt.plot(times, signal1)
plt.plot(times, signal2)
plt.show()

We check the eigenvalues to make sure that there is no resonance while idling.

In [ ]:
matrix = hamiltonian.getMatrixOneTime(0.0)
evals = np.linalg.eigvalsh(matrix)
print("Energies in GHz: ", np.round(evals / 1e6) / 1e3 / (2*np.pi))
transitions = evals[1:] - evals[:-1]
print("Transition energies in GHz: ", np.round(transitions / 1e3) / 1e6 / (2*np.pi))
matrix

We select a propagation method, piecewise constant exponentation, and configure CR as a target gate. Also we initialize the full basis at time 0 with $\mathbb{I}_4$ in order to compute the full propagator. The `restricSubsystem` method has been used to truncate the Hilbert space to 4 dimensions to compare the propagator to the ideal gate.

In [ ]:
model = ClosedModel(hamiltonian)
prop = ScipyExpmGOAT(model, res=100e9)

X = np.array([[0.0, 1], [1, 0.0]])
Y = np.array([[0.0,-1j], [1j, 0.0]])
Z = np.array([[1, 0], [0.0, -1]])

IX = np.kron(np.identity(2), X)
IY = np.kron(np.identity(2), Y)
IZ = np.kron(np.identity(2), Z)

ZX = np.exp(1j*np.pi/4) * np.kron(Z, X)

CRGate = np.array(
    [[1.0, 0, 0, 0],
     [0, 1.0, 0, 0],
     [0, 0, 0, 1.0],
     [0, 0, 1.0, 0]]
)  

CRGate = ZX @ CRGate

prop.setInitialState(np.identity(transmon1.dimension() * transmon2.dimension()))
gateFid = UnitaryFidelity(
    propagation=prop,
    gate=CRGate,
    times=np.array([0.0, t_final]),
)
gateFid.restrictSubsystems([transmon1.dimension(), transmon2.dimension()], [2,2])
gateFid.measure()

In [ ]:
def plotPopulation(propagation: Propagation):
    basis1 = [i for i in range(transmon1.dimension())]
    basis2 = [i for i in range(transmon2.dimension())]
    labels = [f"{i},{j}" for (i, j) in itertools.product(basis1, basis2)]
    
    signal1 = generator1.generateSignal(times)
    signal2 = generator2.generateSignal(times)
    states = propagation.propagate(times)

    fig, ax = plt.subplots(3, figsize=(4,6), sharex=True)
    ax[0].plot(times / 1e-9, signal1)
    ax[0].plot(times / 1e-9, signal2)
    ax[0].set_xlabel("Time [ns]")
    ax[0].set_ylabel("Signal [Hz]")
    
    ax[1].plot(times/1e-9, np.abs(states)[:,:,0] ** 2, label=labels)
    ax[1].set_xlabel("Time [ns]")
    ax[1].set_ylabel("Population")
    ax[1].legend(ncols=2)
    
    ax[2].plot(times/1e-9, np.abs(states)[:,:,transmon2.dimension()] ** 2, label=labels)
    ax[2].set_xlabel("Time [ns]")
    ax[2].set_ylabel("Population")
    ax[2].legend(ncols=2)
    plt.tight_layout()
    plt.show()

plotPopulation(prop)

In [ ]:
def expecationValue(Op, states):
    ex = []
    for state in states:
        state = gateFid._preprocessVector(state)
        ex.append(np.real(state.conj() @ Op @ state.T))
    return ex

def plotPauli():
    states = prop.propagate(times)
    sig1 = generator1.generateSignal(times)
    sig2 = generator2.generateSignal(times)

    fig, ax = plt.subplots(3, figsize=(4,6), sharex=True)
    ax[0].plot(times/1e-9, sig1)
    ax[0].plot(times/1e-9, sig2)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(times/1e-9, expecationValue(IX, states[:,:,0]))
    ax[1].plot(times/1e-9, expecationValue(IY, states[:,:,0]))
    ax[1].plot(times/1e-9, expecationValue(IZ, states[:,:,0]))
    ax[1].set_ylabel(r"$\langle 0, x|\hat\sigma_i|0, x\rangle$")
    ax[1].legend(["X", "Y", "Z"])
    ax[2].plot(times/1e-9, expecationValue(IX, states[:,:,transmon2.dimension()]))
    ax[2].plot(times/1e-9, expecationValue(IY, states[:,:,transmon2.dimension()]))
    ax[2].plot(times/1e-9, expecationValue(IZ, states[:,:,transmon2.dimension()]))
    ax[2].set_ylabel(r"$\langle 1, x|\hat\sigma_i|1, x\rangle$")
    ax[-1].set_xlabel("Time [ns]")
    ax[2].legend(["X", "Y", "Z"])
    return fig, ax

plotPauli()

We define an optimizer and link our fidelity measure as a goal function. The only optimisable parameter is the frequency of transmon 1.

In [ ]:
optmap = OptimisationMap()
optmap.add(tone1)
print(optmap)

opt = ScipyOptimiserGradient(gateFid, optimisables=optmap)
opt.setOptions({"ftol": 0.1})

In [ ]:
optmap

In [ ]:
opt.optimise()

In [ ]:
plotPopulation(prop)

In [ ]:
plotPauli()